In [ ]:
from agents import Agent, trace, Runner,OpenAIChatCompletionsModel, function_tool
from dotenv import load_dotenv
from openai import AsyncOpenAI
from openai import OpenAI
import os
import asyncio

In [6]:
client=AsyncOpenAI(api_key=os.getenv('GOOGLE_GEMINI'), base_url="https://generativelanguage.googleapis.com/v1beta/openai/") 

model = OpenAIChatCompletionsModel(
    model="gemini-2.5-flash-lite",
    openai_client=client
)

In [7]:
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

In [ ]:
message='Write a cold email'

sale_agent1=Agent(
    name='sale agent 1',
    instructions=instructions1,
    model=model
)

sale_agent2=Agent(
    name='sale agent 2',
    instructions=instructions2,
    model=model
)

sale_agent3=Agent(
    name='sale agent 3',
    instructions=instructions3,
    model=model
)

sales_picker = Agent(
    name="sales_picker",
    instructions="You pick the best cold sales email from the given options. \
Imagine you are a customer and pick the one you are most likely to respond to. \
Do not give an explanation; reply with the selected email only.",
    model=model
)

with trace('Fetch the best email response'):
    outputs= await asyncio.gather(
        Runner.run(sale_agent1, message),
        Runner.run(sale_agent2, message),
        Runner.run(sale_agent3, message)
    )

    result=[output.final_output for output in outputs]

    join_result="Cold sales email:\n"+"Email:\n\n".join(result)

    best_email=await Runner.run(sales_picker, join_result)

    print(f"Best sales email:\n{best_email.final_output}")

Error getting response; filtered.input=[{'content': 'Cold sales email:\nSubject: Streamlining SOC 2 Compliance with ComplAI\n\nDear [Name],\n\nMy name is [Your Name] and I am reaching out from ComplAI. We specialize in leveraging artificial intelligence to simplify and automate the complex process of achieving and maintaining SOC 2 compliance.\n\nIn today\'s landscape, demonstrating robust security and data protection is paramount. For organizations like [Company Name], a successful SOC 2 audit is not just a compliance requirement, but a critical factor in building trust with clients and partners.\n\nComplAI offers a comprehensive SaaS solution designed to:\n\n*   **Automate Evidence Collection:** Significantly reduce the manual effort and time required to gather necessary documentation for audits.\n*   **Continuously Monitor Controls:** Proactively identify and address potential compliance gaps before they become issues.\n*   **Generate Audit-Ready Reports:** Provide clear, concise re

RateLimitError: Error code: 429 - [{'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 56.613706725s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash